# ChronicCare Nour — Risk Stratification Decision Tree

Trains a Decision Tree classifier that categorises chronic disease patients into
**low / medium / high** risk based on clinical features extracted from the EHR and
the Gemini conversation layer.

| Feature | Description |
|---|---|
| `age` | Patient age |
| `bmi` | Body Mass Index |
| `hba1c_last` | Last recorded HbA1c (%) |
| `baseline_glucose` | Fasting glucose (mmol/L) |
| `current_glucose` | Glucose from latest conversation |
| `symptom_count` | Number of symptoms extracted by Gemini |
| `has_hypertension` | Co-morbidity flag |
| `has_heart_disease` | Co-morbidity flag |
| `medication_count` | Number of active medications |

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

RNG = np.random.default_rng(42)
N = 2_000

FEATURE_NAMES = [
    "age", "bmi", "hba1c_last", "baseline_glucose", "current_glucose",
    "symptom_count", "has_hypertension", "has_heart_disease", "medication_count",
]
RISK_LABELS = ["low", "medium", "high"]
MODEL_PATH = Path("../models/risk_tree.pkl")

## 1. Generate Synthetic Clinical Dataset

In [ ]:
age               = RNG.integers(25, 80, size=N).astype(float)
bmi               = RNG.uniform(18.5, 45.0, size=N)
hba1c             = RNG.uniform(5.5, 12.0, size=N)
baseline_glucose  = RNG.uniform(4.0, 15.0, size=N)
current_glucose   = np.clip(baseline_glucose + RNG.uniform(-1.5, 3.0, size=N), 3.0, 18.0)
symptom_count     = RNG.integers(0, 7, size=N).astype(float)
has_hypertension  = RNG.integers(0, 2, size=N).astype(float)
has_heart_disease = RNG.integers(0, 2, size=N).astype(float)
medication_count  = RNG.integers(0, 6, size=N).astype(float)

# Clinical rule-based scoring
score = (
    np.where(hba1c >= 9.0, 2, np.where(hba1c >= 7.0, 1, 0))
    + np.where(current_glucose >= 11.1, 2, np.where(current_glucose >= 7.8, 1, 0))
    + np.where(bmi >= 35.0, 1, 0)
    + np.where(age >= 65, 1, 0)
    + has_hypertension
    + has_heart_disease
    + np.where(symptom_count >= 3, 1, 0)
)
label = np.where(score >= 5, 2, np.where(score >= 2, 1, 0))
# 5 % label noise
mask = RNG.random(N) < 0.05
label[mask] = RNG.integers(0, 3, size=mask.sum())

df = pd.DataFrame(dict(
    age=age, bmi=bmi, hba1c_last=hba1c, baseline_glucose=baseline_glucose,
    current_glucose=current_glucose, symptom_count=symptom_count,
    has_hypertension=has_hypertension, has_heart_disease=has_heart_disease,
    medication_count=medication_count, risk_level=label,
))
df["risk_label"] = df["risk_level"].map({0: "low", 1: "medium", 2: "high"})
df["risk_label"].value_counts()

## 2. Train / Evaluate

In [ ]:
X = df[FEATURE_NAMES].to_numpy()
y = df["risk_level"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

clf = DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, class_weight="balanced", random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
print(classification_report(y_test, y_pred, target_names=RISK_LABELS))

## 3. Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=RISK_LABELS, ax=ax)
ax.set_title("Risk Stratification — Confusion Matrix")
plt.tight_layout()
plt.show()

## 4. Feature Importances

In [ ]:
importances = clf.feature_importances_
order = np.argsort(importances)[::-1]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([FEATURE_NAMES[i] for i in order], importances[order])
ax.set_xlabel("Feature")
ax.set_ylabel("Importance")
ax.set_title("Decision Tree — Feature Importances")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

## 5. Visualise Tree

In [ ]:
fig, ax = plt.subplots(figsize=(20, 8))
plot_tree(clf, feature_names=FEATURE_NAMES, class_names=RISK_LABELS,
          filled=True, max_depth=3, ax=ax, fontsize=9)
ax.set_title("Decision Tree (depth ≤ 3)")
plt.tight_layout()
plt.show()

## 6. Save Model

In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(clf, MODEL_PATH)
print(f"Saved → {MODEL_PATH.resolve()}")

## 7. Inference Demo

In [ ]:
import sys
sys.path.insert(0, str(Path("../")))

from src.domain.models.risk_models import RiskFeatures
from src.infrastructure.ml.risk_classifier import RiskClassifier

rc = RiskClassifier()

patients = [
    RiskFeatures(age=45, bmi=26.0, hba1c_last=6.2, baseline_glucose=5.5),
    RiskFeatures(age=58, bmi=31.0, hba1c_last=8.1, baseline_glucose=9.0, symptom_count=2),
    RiskFeatures(age=72, bmi=38.0, hba1c_last=11.5, baseline_glucose=13.0,
                 current_glucose=15.0, symptom_count=5,
                 has_hypertension=True, has_heart_disease=True, medication_count=4),
]

for i, p in enumerate(patients, 1):
    pred = rc.predict(p)
    print(f"Patient {i}: risk={pred.risk_level} ({pred.confidence:.0%}), top={pred.top_features}")